# Arc tokenizer: reference vs reconstruction, as video

Two action chunks for the same moment, drawn on the same frames:

* **A - reference** (green): the ordinary time-based action chunk, truncated at
  the first `D = 0.40 m` of travel. This is what the arm actually did.
* **B - reconstruction** (red): `detokenize(tokenize(chunk))` through the real
  pipeline objects, reconstructed to the same number of control steps as A.

B is the best any arc policy could do. The gap between them is the cost of the
representation, before a model is involved.

Renders three videos - A alone, B alone, both overlaid - plus the numeric error.

**One chunk row is not 1/30 s.** `InterpolatePose` compresses the keymap horizon
into `new_chunk_length` rows, so a row is `(horizon / new_chunk_length) / 30`
seconds: 2/30 for yam, 6/30 for human. The tokenizer is hardcoded to `dt=1/30`,
which makes the stored velocity 2x / 6x too large in absolute terms. It cancels
inside `detokenize` so the reconstruction here is unaffected, but do not read
the velocity token as a physical speed without rescaling.

In [ ]:
import os, pathlib
# run from the repo root no matter where the kernel started
_r = pathlib.Path.cwd()
if _r.name == "notebooks": os.chdir(_r.parent)
print("cwd:", os.getcwd())

import numpy as np, torch, hydra, copy, matplotlib.pyplot as plt
from omegaconf import OmegaConf
from egomimic.rldb.embodiment.embodiment import Embodiment
from egomimic.rldb.zarr.arc_length_tokenizer import (
    TokenizeBimanualArcLengthCartesian, cumulative_arc_length, _dist_interval_indices)

RUN = "logs/abc_mecka_fold_multitask_cotrain_arcD40M100/arcD40M100_2026-09-05_13-14-13/0/.hydra/config.yaml"
cfg = OmegaConf.load(RUN)
D   = float(cfg.evaluator.min_distance_unit)
M   = int(cfg.evaluator.resampled_vector_length)
RAW = "actions_cartesian_untokenized"
ARM = {"left": 0, "right": 7}
print(f"D = {D} m   M = {M}   waypoint spacing = {D/(M-1)*1000:.2f} mm")

## Pick one episode per embodiment

Everything below is keyed on a single episode each for yam and human, so the
videos are one continuous clip rather than a shuffle across episodes.

In [ ]:
GROUP = "newtask"
tok = TokenizeBimanualArcLengthCartesian(min_distance_unit=D, resampled_vector_length=M)

class Ctx:
    """Everything needed to build and draw chunks for one embodiment."""
    def __init__(self, emb, group=GROUP, episode=None):
        self.emb = emb
        self.ds  = hydra.utils.instantiate(cfg.data.valid_datasets[group][emb], _convert_="all")
        self.tl  = hydra.utils.instantiate(cfg.evaluator.transform_lists[emb], _convert_="all")
        self.viz = hydra.utils.instantiate(cfg.evaluator.viz_func[emb], _convert_="all")
        # index_map is [(episode_hash, frame), ...] -- reading it avoids
        # decoding every sample just to learn which episode each index is in
        imap = self.ds.index_map
        uniq = list(dict.fromkeys(h for h, _ in imap))
        self.episode = episode or uniq[0]
        self.idx = [i for i, (h, _) in enumerate(imap) if h == self.episode]
        self.rows_per_raw = {"yam_bimanual": 2, "human_bimanual": 6}.get(emb, 1)
        self.fps = 30 / self.rows_per_raw
        print(f"{emb}: {len(uniq)} episodes, using {self.episode} "
              f"({len(self.idx)} samples, {self.fps:.0f} fps real time)")

YAM   = Ctx("yam_bimanual")      # Ctx("yam_bimanual", episode="<hash>") to pin another
HUMAN = Ctx("human_bimanual")

## Build A and B

A keeps the crossing row, so it ends just past `D` while the token stops exactly
at `D`. That half-row of slack is real and shows up as a small positive floor on
the final-row error.

In [ ]:
def as_np(x): return x.numpy() if torch.is_tensor(x) else np.asarray(x)

def rows_to_D(pos, dist=D):
    cum = cumulative_arc_length(pos)
    end_s = min(dist, float(cum[-1]))
    _, e = _dist_interval_indices(cum, 0.0, end_s)
    return max(1, int(e)), end_s

def make_pair(sample, arm="right"):
    """A = raw chunk cut at D metres.  B = detok(tok(raw)) to the same length."""
    raw = as_np(sample[RAW])
    k, span = rows_to_D(raw[:, ARM[arm]:ARM[arm]+3])
    A = raw[:k+1]
    token = tok.transform({"actions_cartesian": raw.copy()})["actions_cartesian"]
    B = tok.detokenize(token, action_horizon=len(A))
    return A, B, token, k, span

s0 = YAM.ds[YAM.idx[0]]
A, B, token, k, span = make_pair(s0, "right")
print(f"[{YAM.emb} {YAM.episode[:8]}] rows to travel {D} m: {k}   token span {span:.4f} m")
print(f"A {A.shape}   B {B.shape}   token {token.shape}")

## Draw one frame

`viz_gt_preds` is the exact function the evaluator uses: it colours the `batch`
chunk with **Greens** and the `predictions` chunk with **Reds**, both indexed
along the trajectory (early points are pale, the endpoint is dark). Feeding it
A as batch and B as predictions gives the overlay directly.

In [ ]:
def draw(ctx, sample, chunk_gt, chunk_pred, revert=True):
    """Return an RGB frame with chunk_gt in green and chunk_pred in red."""
    b = {k_: (v.clone() if torch.is_tensor(v) else copy.deepcopy(v)) for k_, v in sample.items()}
    b["actions_cartesian"] = torch.as_tensor(chunk_gt).float()
    for k_ in list(b):                      # add the batch dim the transforms expect
        if torch.is_tensor(b[k_]): b[k_] = b[k_][None]
    b["embodiment"] = torch.as_tensor([sample["embodiment"]])
    pb = {k_: (v.clone() if torch.is_tensor(v) else v) for k_, v in b.items()}
    pb["actions_cartesian"] = torch.as_tensor(chunk_pred).float()[None]
    if revert:                              # detokenized poses -> camera frame
        b  = {**b,  **Embodiment.apply_transform(b,  ctx.tl)}
        pb = {**pb, **Embodiment.apply_transform(pb, ctx.tl)}
    preds = {f"{ctx.emb}_actions_cartesian": pb["actions_cartesian"]}
    return ctx.viz(preds, b)[0]

frame = draw(YAM, s0, A, B)
plt.figure(figsize=(9, 5)); plt.imshow(frame); plt.axis("off")
plt.title("green = A reference (raw, cut at D)   |   red = B detok(tok(chunk))");

## Render the three videos

One frame per dataset sample, so consecutive frames advance through the episode
with a fresh chunk each time - the same cadence the val videos use.

In [ ]:
import torchvision.io as tvio
from pathlib import Path
OUT = Path("notebooks/arc_recon_videos"); OUT.mkdir(parents=True, exist_ok=True)

def render(ctx, seconds=20, arm="right", tag=""):
    # A-only, B-only and overlaid videos for the episode this ctx points at.
    # `seconds` is REAL time: ctx.fps already accounts for a chunk row being
    # 2 raw frames on yam and 6 on human, so 20 s is 300 rows / 100 rows.
    n = int(seconds * ctx.fps)
    fa, fb, fo = [], [], []
    print(f"{ctx.emb}: {seconds}s at {ctx.fps:.0f} fps -> {min(n, len(ctx.idx))} frames")
    for i in ctx.idx[:n]:
        s_ = ctx.ds[i]
        A_, B_, *_ = make_pair(s_, arm)
        fa.append(draw(ctx, s_, A_, A_))
        fb.append(draw(ctx, s_, B_, B_))
        fo.append(draw(ctx, s_, A_, B_))
    stem = tag or f"{ctx.emb}_{ctx.episode[:8]}_{arm}"
    for frames, name in ((fa, "A_reference"), (fb, "B_reconstruction"), (fo, "overlay")):
        p_ = OUT / f"{stem}_{name}.mp4"
        tvio.write_video(str(p_), torch.from_numpy(np.ascontiguousarray(np.stack(frames))),
                         fps=ctx.fps)
        print("wrote", p_)

SECONDS = 20
render(YAM,   seconds=SECONDS, arm="right")
render(HUMAN, seconds=SECONDS, arm="right")

## Error between them

In [ ]:
def errors(A, B, arm="right"):
    o = ARM[arm]
    e = np.linalg.norm(A[:, o:o+3] - B[:, o:o+3], axis=1)
    return e, {"mean_mm": e.mean()*1000, "max_mm": e.max()*1000, "final_mm": e[-1]*1000,
               "span_A_m": float(cumulative_arc_length(A[:, o:o+3])[-1]),
               "span_B_m": float(cumulative_arc_length(B[:, o:o+3])[-1])}

e, st = errors(A, B, "right")
for k_, v in st.items(): print(f"  {k_:10s} {v:8.3f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(A[:, 7], A[:, 8], "o-", label="A reference"); ax[0].plot(B[:, 7], B[:, 8], "s--", label="B reconstruction")
ax[0].set_xlabel("x (m)"); ax[0].set_ylabel("y (m)"); ax[0].legend(); ax[0].set_aspect("equal","datalim")
ax[1].plot(e*1000, "o-"); ax[1].axhline(D/(M-1)*1000, ls=":", c="r", label="waypoint spacing")
ax[1].set_xlabel("chunk row"); ax[1].set_ylabel("error (mm)"); ax[1].legend()
for a in ax: a.grid(alpha=.3)
fig.tight_layout();

## Sweep the split

In [ ]:
def sweep(ds, n=150, arms=("left","right")):
    out=[]
    for i in np.linspace(0, len(ds)-1, min(n,len(ds))).astype(int):
        s = ds[int(i)]
        for arm in arms:
            A,B,_,k,span = make_pair(s, arm)
            if len(A) < 3 or span < 1e-3: continue
            _, st = errors(A,B,arm); st.update(rows=len(A), arm=arm); out.append(st)
    return out

res = sweep(YAM.ds)
col = lambda n: np.array([r[n] for r in res])
print(f"{YAM.emb}: {len(res)} arm-samples")
for nm in ("mean_mm","max_mm","final_mm","rows"):
    v=col(nm); print(f"  {nm:9s} p10 {np.percentile(v,10):7.2f}  med {np.median(v):7.2f}  p90 {np.percentile(v,90):7.2f}")

fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].hist(col("mean_mm"), bins=40); ax[0].axvline(D/(M-1)*1000, ls=":", c="r")
ax[0].set_xlabel("mean reconstruction error (mm)")
ax[1].scatter(col("rows"), col("mean_mm"), s=12, alpha=.6)
ax[1].set_xlabel("rows to cover D"); ax[1].set_ylabel("mean error (mm)")
for a in ax: a.grid(alpha=.3)
fig.tight_layout();

## Notes

* Green is A, red is B, and both fade from pale at the start to saturated at the
  endpoint. On a light background the early points are nearly invisible, so
  counting dots by eye undercounts badly - trust the error plot, not the frame.
* Compare the median error against the waypoint spacing `D/(M-1)` = 4.04 mm. Well
  under it means the token is not the limiting factor; above it means `M` is too
  coarse for this motion.
* Rotation and gripper are not scored. Add ypr as a geodesic angle if you want
  them - do not sum radians and metres into one number.
* This is the floor a policy inherits. Set it against
  `Valid/..._detok_paired_mse_avg` from a real run: model error near this floor
  means the representation is the bottleneck, not the model.